<a href="https://colab.research.google.com/github/alearecuest/anyoneai-exercises-sprint_3/blob/main/2_output_parsers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LangChain: Output Parsers

### **Notebook Overview**

This notebook demonstrates how to extract **structured data** from LLM responses using output parsers. Instead of receiving raw text, you will learn to parse LLM outputs into structured formats like dictionaries and validated objects, making them easier to use in downstream applications.

### **What You Will Learn**

* How to define structured output schemas with ResponseSchema
* Creating and using StructuredOutputParser for named fields
* Building type-safe outputs with Pydantic models
* Integrating output parsers into LangChain chains
* Handling parsing errors with small models

### **Key Concepts Covered**

* Structured output extraction from LLMs
* ResponseSchema for field definitions
* Pydantic models for type validation
* Format instructions and prompt engineering for structured responses
* Trade-offs between model size and structured output reliability

**Important:** For demonstration purposes and simplicity, we use a small model (0.5B parameters). These smaller models often do **not** follow instructions reliably. **For more consistent results, switch to a larger model or a private one such as GPT-5.**

## 1. Setup

Install dependencies and load the model:

In [ ]:
# Install required packages
%pip install langchain langchain-huggingface --quiet

# Alternative for OpenAI:
# %pip install langchain langchain-openai --quiet

In [ ]:
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Load Qwen 2.5 Instruct 0.5B model
model_id = "Qwen/Qwen2.5-0.5B-Instruct"

print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    dtype="auto"
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.3,
    return_full_text=False,
    do_sample=False # Do sample False (Is like 0 temperature)
)

llm = HuggingFacePipeline(pipeline=pipe)
print("✅ Model ready!")

# Alternative for OpenAI:
# from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(model="gpt-5-mini", temperature=0.3)

Loading model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Device set to use cpu
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✅ Model ready!


## 4. Structured Output Parser

Get structured data with named fields:

In this case the expected output is a JSON file with the following format:

```json
{
    "name": "The name of the person",
    "profession": "The person's profession",
    "achievement": "A notable achievement"
}
```


In [ ]:
from langchain.output_parsers import StructuredOutputParser, ResponseSchema
from langchain.prompts import PromptTemplate

# Define the structure we want
response_schemas = [
    ResponseSchema(name="name", description="The name of the person"),
    ResponseSchema(name="profession", description="The person's profession"),
    ResponseSchema(name="achievement", description="A notable achievement")
]

# Create parser
structured_parser = StructuredOutputParser.from_response_schemas(response_schemas)

# Get format instructions
format_instructions = structured_parser.get_format_instructions()
print("Format instructions:")
print(format_instructions)
print("\n" + "="*50 + "\n")

# Create prompt
structured_prompt = PromptTemplate(
    input_variables=["person"],
    template="Provide information about {person}.\n{format_instructions}",
    partial_variables={"format_instructions": format_instructions}
)

# Create chain
structured_chain = structured_prompt | llm | structured_parser

# Use it
try:
    result = structured_chain.invoke({"person": "Marie Curie"})
    print(f"Type: {type(result)}")
    print(f"Result: {result}")
    print(f"\nName: {result.get('name', 'N/A')}")
    print(f"Profession: {result.get('profession', 'N/A')}")
except Exception as e:
    print(f"Note: Small models may struggle with complex structured output.")
    print(f"Error: {e}")

Format instructions:
The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"name": string  // The name of the person
	"profession": string  // The person's profession
	"achievement": string  // A notable achievement
}
```


Type: <class 'dict'>
Result: {'name': 'Marie Curie', 'profession': 'Chemist', 'achievement': 'In 1903, she discovered two new elements: polonium (named after her) and radium. She was awarded the Nobel Prize in Physics for these discoveries.'}

Name: Marie Curie
Profession: Chemist


## 5. Pydantic Output Parser

Use Pydantic models for type-safe structured output:

In this case the expected output is a JSON file with the following format:

```json
{
    "title": "The title of the book",
    "author": "The author's name",
    "year": "Publication year"
}
```

In [ ]:
from langchain.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# Define a Pydantic model
class Book(BaseModel):
    title: str = Field(description="The title of the book")
    author: str = Field(description="The author's name")
    year: int = Field(description="Publication year")

# Create parser
pydantic_parser = PydanticOutputParser(pydantic_object=Book)

# Get format instructions
format_instructions = pydantic_parser.get_format_instructions()
print("Format instructions:")
print(format_instructions)
print("\n" + "="*50 + "\n")

# Create prompt
pydantic_prompt = PromptTemplate(
    input_variables=["book_description"],
    template="Extract book information from: {book_description}\n{format_instructions}",
    partial_variables={"format_instructions": format_instructions}
)

# Create chain
pydantic_chain = pydantic_prompt | llm | pydantic_parser

# Use it
try:
    result = pydantic_chain.invoke({
        "book_description": "Deep Learning by Ian Goodfellow, Yoshua Bengio, and Aaron Courville (2016), a comprehensive textbook on modern neural networks."
    })
    print(f"Type: {type(result)}")
    print(f"Result: {result}")
    print(f"\nTitle: {result.title}")
    print(f"Author: {result.author}")
    print(f"Year: {result.year}")
except Exception as e:
    print(f"Note: Small models may struggle with complex JSON output.")
    print(f"Error: {e}")

Format instructions:
The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"title": {"description": "The title of the book", "title": "Title", "type": "string"}, "author": {"description": "The author's name", "title": "Author", "type": "string"}, "year": {"description": "Publication year", "title": "Year", "type": "integer"}}, "required": ["title", "author", "year"]}
```


Type: <class '__main__.Book'>
Result: title='Deep Learning' author='Ian Goodfellow, Yoshua Bengio, and Aaron Courville' year=2016

Title: Deep Learning
Author: Ian Goodfellow, Yoshua Bengio, and Aaron Courville
